# ARCAS body-only transonic CFD diagnostic — SU2 on Colab

Runs actual axisymmetric RANS cases at Mach 0.90, 1.00 and 1.20; meshes, configurations, histories, surface fields, failures and numerical gates are retained in the downloaded ZIP. A clean solver exit is **not** convergence.

Geometry is the production model's generic tangent ogive, with nominal 0.65D rear diameter. It is **not** the exact published contour, and the production fixture retains a 1.50-in rear diameter. Fins are omitted. These cases cannot establish full-rocket CP or be directly fitted to the finned experimental drag.

NASA TN D-4013 supplies Re=9.8×10⁶/m and No. 120 carborundum trip strips. Fully turbulent, adiabatic smooth walls approximate the tripped flow; static T=288 K is a diagnostic assumption, not an acquired tunnel temperature. Resolve geometry and thermal conditions before treating CFD as calibration evidence.

SU2 8.5.0 integrates axisymmetric pressure and friction with 2πr weighting (`CFVMFlowSolverBase.inl`). Coefficients use πD²/4. Both ARCAS CAcorr and NASA/TM-2005-213541 cavity/base-corrected CAF exclude base force; use non-base forces for comparisons. Do not derive CP from this zero-angle axisymmetric run.

Run installation, mesh functions, configuration/assessment, then cases. If a case fails, inspect its logs; the final export cell still downloads the evidence. Three mesh levels and a larger-domain check precede acceptance. Numerical gates are necessary, not proof of experimental accuracy. No production correction is written by this notebook.


In [ ]:
import os, pathlib, subprocess, urllib.request, zipfile
root = pathlib.Path('/content/SU2')
if not (root / 'bin/SU2_CFD').is_file():
    urllib.request.urlretrieve('https://github.com/su2code/SU2/releases/download/v8.5.0/SU2-v8.5.0-linux64-omp.zip', '/tmp/su2.zip')
    with zipfile.ZipFile('/tmp/su2.zip') as release: release.extractall(root)
    with zipfile.ZipFile(root / 'linux64-omp.zip') as binaries: binaries.extractall(root)
su2 = root / 'bin/SU2_CFD'
su2.chmod(su2.stat().st_mode | 0o111)
version = subprocess.run([str(su2), '--help'], check=True, capture_output=True, text=True).stdout
print(version)
import numpy as np
import csv, json, math, hashlib, shutil, time, re, uuid
output_root = pathlib.Path('/content/arcas_reference')
output_root.mkdir(exist_ok=True)
run_root = output_root / ('run_' + time.strftime('%Y%m%dT%H%M%S') + '_' + uuid.uuid4().hex[:8])
run_root.mkdir(exist_ok=False)
print('Evidence for this run:', run_root)


In [ ]:
D = 0.05715; R = D / 2; NOSE = 0.26924; LENGTH = 1.04013; BOAT = 0.045974
BASE_R = 0.65 * R
AREA = math.pi * R * R
RE_PER_M = 9.8e6
TEMPERATURE = 288.0  # Diagnostic assumption, NOT an acquired tunnel temperature.

def graded(length, first, ratio, cap):
    values = [0.0]; step = first
    while values[-1] < length:
        values.append(min(length, values[-1] + step))
        step = min(cap, step * ratio)
    return values

def make_mesh(name, refinement=1.0, domain_scale=1.0):
    height = 50 * D * domain_scale; wake_length = 40 * D * domain_scale
    rho = (R*R + NOSE*NOSE) / (2*R)
    def radius(x):
        if x <= 0: return 0.0
        if x <= NOSE: return math.sqrt(rho*rho - (NOSE-x)**2) - (rho-R)
        if x <= LENGTH-BOAT: return R
        return R + (BASE_R-R)*(x-(LENGTH-BOAT))/BOAT
    def stations(a, b, count):
        return list(np.linspace(a, b, round(count*refinement)+1))[1:]
    xs = [-x for x in reversed(graded(20*D*domain_scale, .0001/refinement, 1.2**(1/refinement), .15/refinement))]
    xs += stations(0, NOSE, 180) + stations(NOSE, LENGTH-BOAT, 300) + stations(LENGTH-BOAT, LENGTH, 80)
    eta = np.array(graded(height, 2e-6/refinement, 1.15**(1/refinement), .15/refinement)) / height
    nodes = []; front = []
    for x in xs:
        r = radius(x); row = []
        for e in eta:
            row.append(len(nodes)); nodes.append((float(x), float(r+(height-r)*e)))
        front.append(row)
    base_count = round(30*refinement)
    base_points = [len(nodes)+i for i in range(base_count)]
    for y in np.linspace(0, BASE_R, base_count+1)[:-1]: nodes.append((LENGTH, float(y)))
    wake = [base_points+front[-1]]
    ys = [nodes[i][1] for i in wake[0]]
    for distance in graded(wake_length, 2e-6/refinement, 1.2**(1/refinement), .08/refinement)[1:]:
        t = distance / wake_length; row = []
        for y in ys:
            row.append(len(nodes)); nodes.append((LENGTH+wake_length*t**(1-.6*y/height), y))
        wake.append(row)
    elements = []
    for rows in (front, wake):
        for a,b in zip(rows, rows[1:]):
            elements += [(a[j],b[j],b[j+1],a[j+1]) for j in range(len(a)-1)]
    markers = {name:[] for name in ['body','base','axis','farfield']}
    for i,(a,b) in enumerate(zip(front, front[1:])):
        markers['body' if xs[i+1]>0 else 'axis'].append((a[0], b[0]))
        markers['farfield'].append((a[-1], b[-1]))
    for a,b in zip(wake, wake[1:]):
        markers['axis'].append((a[0],b[0])); markers['farfield'].append((a[-1],b[-1]))
    markers['base'] = list(zip(wake[0][:base_count], wake[0][1:base_count+1]))
    markers['farfield'] += list(zip(front[0],front[0][1:])) + list(zip(wake[-1],wake[-1][1:]))
    for element in elements:
        q = [nodes[i] for i in element]
        signed_area = .5*sum(q[j][0]*q[(j+1)%4][1]-q[(j+1)%4][0]*q[j][1] for j in range(4))
        if signed_area <= 0: raise ValueError(f'Invalid quad: {element}')
    mesh = run_root / (name+'.su2')
    with mesh.open('w') as f:
        f.write(f'NDIME= 2\nNELEM= {len(elements)}\n')
        for k,e in enumerate(elements): f.write('9 '+' '.join(map(str,e))+f' {k}\n')
        f.write(f'NPOIN= {len(nodes)}\n')
        for k,(x,y) in enumerate(nodes): f.write(f'{x:.16g} {y:.16g} {k}\n')
        f.write(f'NMARK= {len(markers)}\n')
        for tag,edges in markers.items():
            f.write(f'MARKER_TAG= {tag}\nMARKER_ELEMS= {len(edges)}\n')
            for a,b in edges: f.write(f'3 {a} {b}\n')
    meta = {'nodes':len(nodes),'quads':len(elements),'refinement':refinement,'domain_scale':domain_scale,
            'mesh_sha256':hashlib.sha256(mesh.read_bytes()).hexdigest(),
            'first_wall_spacing_m':2e-6/refinement,'body_edges':markers['body'],'base_edges':markers['base'],
            'body_node_coords':{str(i):nodes[i] for edge in markers['body'] for i in edge}}
    (run_root/(name+'.json')).write_text(json.dumps(meta,indent=2))
    print(name, meta['nodes'], 'nodes', meta['quads'], 'quads')
    return mesh, meta


In [ ]:
TURBULENCE = 'SST'
MACHES = [0.90, 1.00, 1.20]

def config(mach, mesh, iterations, second_order=False, restart=False):
    return f"""SOLVER= RANS
KIND_TURB_MODEL= {TURBULENCE}
AXISYMMETRIC= YES
MATH_PROBLEM= DIRECT
RESTART_SOL= {'YES' if restart else 'NO'}
SOLUTION_FILENAME= restart.dat
MACH_NUMBER= {mach}
AOA= 0.0
INIT_OPTION= REYNOLDS
REYNOLDS_NUMBER= {RE_PER_M}
REYNOLDS_LENGTH= 1.0
FREESTREAM_TEMPERATURE= {TEMPERATURE}
REF_LENGTH= {LENGTH}
REF_AREA= {AREA}
MARKER_HEATFLUX= ( body, 0.0, base, 0.0 )
MARKER_FAR= ( farfield )
MARKER_SYM= ( axis )
MARKER_MONITORING= ( body, base )
MARKER_PLOTTING= ( body, base )
MESH_FILENAME= {mesh}
MESH_FORMAT= SU2
NUM_METHOD_GRAD= WEIGHTED_LEAST_SQUARES
CONV_NUM_METHOD_FLOW= ROE
MUSCL_FLOW= {'YES' if second_order else 'NO'}
SLOPE_LIMITER_FLOW= VENKATAKRISHNAN
TIME_DISCRE_FLOW= EULER_IMPLICIT
CONV_NUM_METHOD_TURB= SCALAR_UPWIND
TIME_DISCRE_TURB= EULER_IMPLICIT
CFL_NUMBER= 0.5
CFL_ADAPT= NO
LINEAR_SOLVER= FGMRES
LINEAR_SOLVER_PREC= ILU
LINEAR_SOLVER_ITER= 50
MGLEVEL= 0
ITER= {iterations}
CONV_FIELD= RMS_DENSITY
CONV_RESIDUAL_MINVAL= -8
CONV_STARTITER= 10
SCREEN_OUTPUT= (INNER_ITER, RMS_DENSITY, RMS_ENERGY, DRAG)
HISTORY_OUTPUT= (ITER, RMS_RES, AERO_COEFF, AERO_COEFF_SURF)
VOLUME_OUTPUT= (COORDINATES, SOLUTION, PRIMITIVE)
SCREEN_WRT_FREQ_INNER= 100
OUTPUT_FILES= (RESTART, SURFACE_TECPLOT_ASCII)
SURFACE_FILENAME= surface_fields
OUTPUT_WRT_FREQ= 500
"""

def read_csv(path):
    with path.open() as f:
        rows = list(csv.DictReader(f,skipinitialspace=True))
    return [{k.strip():float(v) for k,v in row.items()} for row in rows]

def read_tecplot_surface(path):
    """Read SU2's single-zone, POINT-packed ASCII Tecplot wall surface output."""
    lines = path.read_text().splitlines()
    variable_line = next((line for line in lines if line.strip().upper().startswith('VARIABLES')), None)
    if variable_line is None:
        raise ValueError(f'No VARIABLES declaration in {path}')
    names = re.findall(r'"([^"]+)"|([^,\s=]+)', variable_line.split('=',1)[1])
    names = [quoted or bare for quoted,bare in names]
    zone_index = next((i for i,line in enumerate(lines) if line.strip().upper().startswith('ZONE')), None)
    if zone_index is None:
        raise ValueError(f'No ZONE declaration in {path}')
    zone = lines[zone_index].upper()
    if 'DATAPACKING=POINT' not in zone.replace(' ', '') and 'F=POINT' not in zone.replace(' ', ''):
        raise ValueError(f'Expected POINT-packed Tecplot data, got: {lines[zone_index]}')
    match = re.search(r'\bNODES\s*=\s*(\d+)', zone)
    expected = int(match.group(1)) if match else None
    values = []
    for line in lines[zone_index+1:]:
        stripped = line.strip()
        if not stripped or stripped.startswith('#'):
            continue
        if stripped.upper().startswith('ZONE'):
            break
        try:
            row = [float(token) for token in stripped.replace(',', ' ').split()]
        except ValueError:
            continue
        values.extend(row)
    if len(values) % len(names):
        raise ValueError(f'Tecplot value count {len(values)} is not divisible by {len(names)} variables')
    rows = [dict(zip(names, values[i:i+len(names)])) for i in range(0,len(values),len(names))]
    if expected is not None and len(rows) != expected:
        raise ValueError(f'Tecplot declared {expected} nodes but contained {len(rows)} rows')
    required = {'x','y','pressure_coefficient','skin_friction_coefficient_x','y_plus'}
    normalized = {key.lower() for key in rows[0]} if rows else set()
    missing = required - normalized
    if missing:
        raise ValueError(f'Tecplot surface fields missing {sorted(missing)}; found {list(rows[0]) if rows else []}')
    # Normalize names for downstream code; retain all fields such as Y_Plus.
    return [{k.lower():v for k,v in row.items()} for row in rows]

def associate_surface_nodes(surface_rows, mesh_meta):
    """Map mesh node ids to Tecplot rows using nearest coordinates within tolerance."""
    node_ids = sorted(int(i) for i in mesh_meta['body_node_coords'])
    mesh_xy = [mesh_meta['body_node_coords'][str(i)] for i in node_ids]
    surface_xy = [(r['x'],r['y']) for r in surface_rows]
    if not surface_xy: raise ValueError('Tecplot surface zone is empty')
    # Tecplot coordinates are rounded; use a physical tolerance, never float equality.
    coordinate_scale = max(LENGTH,D,max(abs(value) for xy in mesh_xy for value in xy))
    # SU2 Tecplot uses about seven significant digits; scale the tolerance
    # to the largest coordinate so aft-body rounding remains matchable.
    tolerance = 1e-6 * coordinate_scale
    nearest = []
    errors = []
    for x,y in mesh_xy:
        distances = [math.hypot(x-sx,y-sy) for sx,sy in surface_xy]
        index = min(range(len(distances)),key=distances.__getitem__)
        nearest.append(index); errors.append(distances[index])
    if max(errors) > tolerance:
        raise ValueError(f'{sum(error>tolerance for error in errors)} body mesh nodes lack a Tecplot coordinate match within {tolerance:g} m (worst {max(errors):g} m)')
    if len(set(nearest)) != len(nearest):
        raise ValueError('Multiple body mesh nodes matched the same Tecplot point; coordinate precision is insufficient')
    return {node_id:surface_rows[row_index] for node_id,row_index in zip(node_ids,nearest)}

def marker_drag(row, marker):
    keys = [k for k in row if k.startswith('CD') and marker in k]
    if len(keys) != 1: raise ValueError(f'Missing/ambiguous {marker} drag column: {list(row)}')
    return row[keys[0]]

def assess(case, mesh_meta):
    history = read_csv(case/'second_order_history.csv')
    surface = read_tecplot_surface(case/'surface_fields.dat')
    points = associate_surface_nodes(surface,mesh_meta)
    if len(history) < 300: raise ValueError('Need at least 300 force-history samples')
    last = history[-1]; window = history[-300:]
    cd_values = [r['CD'] for r in window]
    mean_cd = sum(cd_values)/len(cd_values)
    relative_span = float((max(cd_values)-min(cd_values)) / max(abs(mean_cd),1e-8))
    initial = read_csv(case/'initialize_history.csv')[0]
    density_drop = initial['rms[Rho]'] - last['rms[Rho]']
    energy_drop = initial['rms[RhoE]'] - last['rms[RhoE]']
    pressure_body = 0.0; friction_body = 0.0
    for a,b in mesh_meta['body_edges']:
        p,q = points[a],points[b]; dr = q['y']-p['y']; dx = q['x']-p['x']
        ring = math.pi*(p['y']+q['y'])
        pressure_body += .5*(p['pressure_coefficient']+q['pressure_coefficient'])*dr*ring/AREA
        friction_body += .5*(p['skin_friction_coefficient_x']+q['skin_friction_coefficient_x'])*math.hypot(dx,dr)*ring/AREA
    body_cd = marker_drag(last,'body'); base_cd = marker_drag(last,'base')
    force_closure = abs(body_cd+base_cd-last['CD']) / max(abs(last['CD']),1e-8)
    integration_error = abs(pressure_body+friction_body-body_cd) / max(abs(body_cd),1e-8)
    yplus = max(r['y_plus'] for r in surface)
    finite = all(math.isfinite(r[k]) for r in history for k in ['CD','rms[Rho]','rms[RhoE]'])
    if not finite or not all(math.isfinite(v) for v in [body_cd, base_cd, pressure_body, friction_body, yplus, relative_span, density_drop, energy_drop]):
        raise ValueError('Non-finite force, wall diagnostic or residual history')
    accepted = finite and last['rms[Rho]']<=-6 and density_drop>=3 and energy_drop>=3 and relative_span<=.001 and yplus<=2 and force_closure<=.001 and integration_error<=.02
    result = {'accepted_numerics':accepted,'CD_total':last['CD'],'CD_body_nonbase':body_cd,'CD_base':base_cd,
              'CD_body_pressure_integrated':pressure_body,'CD_body_friction_integrated':friction_body,
              'density_residual':last['rms[Rho]'],'density_drop_decades':density_drop,'energy_drop_decades':energy_drop,
              'last_300_drag_relative_span':relative_span,'max_yplus':yplus,'marker_force_closure':force_closure,
              'surface_integration_error':integration_error,'last_iteration':last['Inner_Iter']}
    return result


In [ ]:
results = []
def run_case(mach, mesh_name, mesh, meta):
    case = run_root / f'M{mach:.2f}_{mesh_name}_{TURBULENCE}'
    case.mkdir(exist_ok=True)
    record = {'mach':mach,'mesh':mesh_name,'turbulence':TURBULENCE,'geometry':'generic tangent-ogive ARCAS body only; nominal 0.65D base',
              'Re_per_meter':RE_PER_M,'static_temperature_K':TEMPERATURE,'temperature_source':'diagnostic assumption',
              'reference_area_m2':AREA,'axisymmetric':True,'mesh_metadata':meta,'solver_help':version,'started':time.time()}
    try:
        for stage,iterations,second,restart in [('initialize',2000,False,False),('second_order',6000,True,True)]:
            cfg_path = case/(stage+'.cfg')
            cfg_path.write_text(config(mach,mesh,iterations,second,restart))
            with (case/(stage+'.log')).open('w') as f:
                exit_code = subprocess.run([str(su2),str(cfg_path),'-t','2'],cwd=case,stdout=f,stderr=subprocess.STDOUT).returncode
            if (case/'history.csv').exists(): shutil.copyfile(case/'history.csv',case/(stage+'_history.csv'))
            if exit_code: raise RuntimeError(f'{stage} failed with exit {exit_code}; inspect retained log')
        record.update(assess(case,meta))
    except Exception as exc:
        record.update(accepted_numerics=False,error=str(exc))
    record['finished'] = time.time()
    (case/'result.json').write_text(json.dumps(record,indent=2,allow_nan=False))
    results.append(record)
    print({k:v for k,v in record.items() if k not in ['mesh_metadata','solver_help']}, flush=True)
    if not record['accepted_numerics']: raise RuntimeError(f'Rejected numerical case {case}; do not use for corrections')

# Three mesh levels at every Mach, then an enlarged-domain comparison at M=0.90.
for name,refinement,domain_scale,maches in [('coarse',1,1,MACHES),('medium',1.5,1,MACHES),('fine',2,1,MACHES),('domain',2,1.5,[.90])]:
    mesh,meta = make_mesh(name,refinement,domain_scale)
    for mach in maches: run_case(mach,name,mesh,meta)


In [ ]:
# Run this cell even when a numerical case was rejected: failed-run evidence matters.
by_key = {(r['mach'],r['mesh']):r for r in results}
sensitivity = []
for mach in MACHES:
    labels = ['coarse','medium','fine'] + (['domain'] if mach==.90 else [])
    records = [by_key.get((mach,label)) for label in labels]
    entry = {'mach':mach,'accepted_reference':False}
    if all(r and r['accepted_numerics'] for r in records):
        fine = by_key[(mach,'fine')]['CD_body_nonbase']
        changes = [abs(r['CD_body_nonbase']-fine)/max(abs(fine),1e-8) for r in records]
        entry.update(max_relative_mesh_domain_change=max(changes),accepted_reference=max(changes)<=.01)
    sensitivity.append(entry)
manifest = {'cases':results,'sensitivity':sensitivity,'scope':'body-only alpha=0 diagnostic; NOT full-rocket drag or CP validation',
            'production_corrections_authorized':False,
            'remaining_experimental_requirements':['match measured contour and rear diameter','acquire tunnel temperature','compare identical non-base observables and validate transfer']}
(run_root/'manifest.json').write_text(json.dumps(manifest,indent=2,allow_nan=False))
archive = shutil.make_archive(str(output_root/run_root.name),'zip',run_root)
print(json.dumps(sensitivity,indent=2))
print('ARTIFACT',archive)
from google.colab import files
files.download(archive)
